# Week 3 — Part 02: Data profiling script (CSV → JSON/Markdown)

**Estimated time:** 90–120 minutes

## Learning Objectives

- Treat real-world CSV data as untrusted input
- Build a deterministic profiling artifact (`profile.json` + `profile.md`)
- Fail fast with clear errors for missing/empty inputs
- Include duplicate detection, numeric statistics, and categorical summaries
- Add optional schema/required-column checks


**Lab tutorial**: [02_data_profiling_script.md](./02_data_profiling_script.md) - Detailed walkthrough and learning objectives

## Overview

In AI/ML/LLM projects, most pain starts with data issues:

- wrong column names
- unexpected types
- empty files
- missing values

A data profiling script makes these issues visible early.

---

## Pre-study (Self-learn)

Self-learn is optional. If you want a refresher on modules, exceptions, file I/O, or JSON:

- [Foundations Course Pre-study index](../PRESTUDY.md)
- [Self-learn — Modules and exception handling](../self_learn/Chapters/2/02_modules_exceptions.md)

---

## What success looks like (end of Part 02)

Given the same input CSV, your code should always produce:

- `output/profile.json` (machine-readable)
- `output/profile.md` (human-readable)

And it should fail with clear errors for:

- missing file
- empty file
- missing required columns (optional extension)

### Checkpoint

After you run the notebook end-to-end, you should see `output/profile.json` and `output/profile.md` on disk.

Key reproducibility detail: keep outputs deterministic so diffs are meaningful.

---

## Output contract (recap)

Given the same input CSV, the script should always produce:

- `output/profile.json`
- `output/profile.md`

### What this cell does
Defines the core data structures and helper functions for the profiling script:

- **`Profile` dataclass** — a typed container for all profile fields (row count, column types, missing values, duplicates, numeric stats, categorical top values). Using a dataclass makes it easy to serialize to JSON with `asdict()`.
- **`load_csv(path)`** — validates the file exists and is non-empty *before* reading it. This is "fail fast" defensive programming: catch problems at the boundary, not deep inside your logic.
- **`make_profile(df)`** — computes the full profile from a DataFrame. Notice `int(...)` and `float(...)` casts — pandas returns numpy types which are not JSON-serializable by default.

**Why `sort_keys=True` matters:** Running the same script twice on the same input should produce byte-for-byte identical JSON. Sorted keys guarantee that — without it, dict ordering can vary between Python versions.

In [ ]:
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List

import pandas as pd


OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


@dataclass
class Profile:
    rows: int
    cols: int
    columns: List[str]
    dtypes: Dict[str, str]
    missing_by_column: Dict[str, int]
    duplicate_rows: int
    numeric_summary: Dict[str, Dict[str, float]]
    categorical_top_values: Dict[str, Dict[str, int]]


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError("Input file not found: %s" % path)
    if path.stat().st_size == 0:
        raise ValueError("Input file is empty: %s" % path)
    return pd.read_csv(path)


def make_profile(df: pd.DataFrame) -> Profile:
    missing = df.isna().sum().to_dict()
    dtypes = {col: str(dtype) for col, dtype in df.dtypes.to_dict().items()}

    numeric_cols = df.select_dtypes(include="number")
    numeric_summary = {}
    for col in numeric_cols.columns:
        numeric_summary[col] = {
            "min": float(numeric_cols[col].min()),
            "max": float(numeric_cols[col].max()),
            "mean": round(float(numeric_cols[col].mean()), 2),
        }

    categorical_cols = df.select_dtypes(include="object")
    categorical_top = {}
    for col in categorical_cols.columns:
        top5 = df[col].value_counts().head(5).to_dict()
        categorical_top[col] = {str(k): int(v) for k, v in top5.items()}

    return Profile(
        rows=int(df.shape[0]),
        cols=int(df.shape[1]),
        columns=list(df.columns),
        dtypes=dtypes,
        missing_by_column={k: int(v) for k, v in missing.items()},
        duplicate_rows=int(df.duplicated().sum()),
        numeric_summary=numeric_summary,
        categorical_top_values=categorical_top,
    )


print("ready")

### What this cell does
Loads the provided sample CSV from `data/sample.csv`. This dataset has intentional data quality issues (missing values in `age`, `city`, `salary` columns, and one duplicate row) so you can see how the profiler handles them.

**Why use a provided dataset?** Starting with known data lets you verify the profiler's output is correct — you know exactly how many missing values and duplicates to expect.

In [ ]:
## Load the provided sample CSV
sample_path = Path("data/sample.csv")
df = load_csv(sample_path)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns from {sample_path}")
df.head()

### What this cell does
Defines `profile_to_markdown()` — converts the `Profile` dataclass into a human-readable Markdown document with tables for columns, numeric stats, and categorical top values — then runs the full pipeline: load → profile → write both `profile.json` and `profile.md`.

**Why two output formats?**
- `profile.json` is machine-readable: downstream scripts can parse it programmatically.
- `profile.md` is human-readable: you can open it in any editor or GitHub to quickly inspect results.

**Reproducibility check:** Run this cell twice on the same input. The output files should be byte-for-byte identical. If they differ, something in your pipeline is non-deterministic.

In [ ]:
def profile_to_markdown(p: Profile) -> str:
    lines = []
    lines.append("# Data Profile")
    lines.append("")
    lines.append(f"- Rows: {p.rows}")
    lines.append(f"- Columns: {p.cols}")
    lines.append(f"- Duplicate rows: {p.duplicate_rows}")
    lines.append("")
    lines.append("## Columns")
    lines.append("")
    lines.append("| column | dtype | missing |")
    lines.append("|---|---|---:|")
    for col in p.columns:
        lines.append(f"| {col} | {p.dtypes.get(col, '')} | {p.missing_by_column.get(col, 0)} |")
    lines.append("")

    if p.numeric_summary:
        lines.append("## Numeric Summary")
        lines.append("")
        lines.append("| column | min | max | mean |")
        lines.append("|---|---:|---:|---:|")
        for col, stats in p.numeric_summary.items():
            lines.append(f"| {col} | {stats['min']} | {stats['max']} | {stats['mean']} |")
        lines.append("")

    if p.categorical_top_values:
        lines.append("## Top Categorical Values")
        lines.append("")
        for col, values in p.categorical_top_values.items():
            lines.append(f"### {col}")
            lines.append("")
            lines.append("| value | count |")
            lines.append("|---|---:|")
            for val, count in values.items():
                lines.append(f"| {val} | {count} |")
            lines.append("")

    return "\n".join(lines)


## Run the full pipeline
p = make_profile(df)
(OUTPUT_DIR / "profile.json").write_text(json.dumps(asdict(p), indent=2, sort_keys=True), encoding="utf-8")
(OUTPUT_DIR / "profile.md").write_text(profile_to_markdown(p), encoding="utf-8")
print("wrote:", OUTPUT_DIR / "profile.json")
print("wrote:", OUTPUT_DIR / "profile.md")

### What this cell does
Defines `require_columns()` — a validator that checks whether all required columns are present in the DataFrame.

**Why validate columns explicitly?** Downstream code often assumes specific column names exist. Without this check, a missing column causes a cryptic `KeyError` deep inside your logic. Validating at the boundary gives a clear, actionable error message.

**Your task:** Try calling `require_columns(df, ["id", "name", "nonexistent"])` and observe the error message.

In [ ]:
from typing import Any


def require_columns(df: pd.DataFrame, required: List[str]) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError("Missing required columns: %s" % missing)


# Test: this should pass
require_columns(df, ["id", "name", "age"])
print("Required columns check passed")

# TODO: try require_columns(df, ["id", "name", "nonexistent"]) and observe the error

## Appendix: Verify the outputs

Quick check that the outputs are correct and the profile captures the expected data quality issues.

In [ ]:
## Verify profile.json was written correctly
with open(OUTPUT_DIR / "profile.json", "r", encoding="utf-8") as f:
    saved_profile = json.load(f)

print(f"Rows: {saved_profile['rows']}")
print(f"Columns: {saved_profile['cols']}")
print(f"Duplicate rows: {saved_profile['duplicate_rows']}")
print(f"Missing values: {saved_profile['missing_by_column']}")
print(f"Numeric columns profiled: {list(saved_profile['numeric_summary'].keys())}")
print(f"Categorical columns profiled: {list(saved_profile['categorical_top_values'].keys())}")

## Verify profile.md was written
md_content = (OUTPUT_DIR / "profile.md").read_text(encoding="utf-8")
print(f"\nprofile.md preview (first 20 lines):")
for line in md_content.split("\n")[:20]:
    print(line)